In [2]:
import pandas as pd

movies = pd.read_csv('data/movies.csv')
ratings = pd.read_csv('data/ratings.csv')

print("Movies shape: ", movies.shape)
print("Ratigs shape: ", ratings.shape)

Movies shape:  (9742, 3)
Ratigs shape:  (100836, 4)


In [3]:
movies.head(10)

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
5,6,Heat (1995),Action|Crime|Thriller
6,7,Sabrina (1995),Comedy|Romance
7,8,Tom and Huck (1995),Adventure|Children
8,9,Sudden Death (1995),Action
9,10,GoldenEye (1995),Action|Adventure|Thriller


In [4]:
ratings.head(10)

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
5,1,70,3.0,964982400
6,1,101,5.0,964980868
7,1,110,4.0,964982176
8,1,151,5.0,964984041
9,1,157,5.0,964984100


In [5]:
print(movies.isnull().sum())
print(ratings.isnull().sum())

movieId    0
title      0
genres     0
dtype: int64
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


In [6]:
print(movies['genres'].head(20))

0     Adventure|Animation|Children|Comedy|Fantasy
1                      Adventure|Children|Fantasy
2                                  Comedy|Romance
3                            Comedy|Drama|Romance
4                                          Comedy
5                           Action|Crime|Thriller
6                                  Comedy|Romance
7                              Adventure|Children
8                                          Action
9                       Action|Adventure|Thriller
10                           Comedy|Drama|Romance
11                                  Comedy|Horror
12                   Adventure|Animation|Children
13                                          Drama
14                       Action|Adventure|Romance
15                                    Crime|Drama
16                                  Drama|Romance
17                                         Comedy
18                                         Comedy
19             Action|Comedy|Crime|Drama|Thriller


In [13]:
import re
all_genres = set()
for g in movies['genres']:
    for genre in g.split('|'):
        all_genres.add(genre)

print(sorted(all_genres))
print(len(sorted(all_genres)))


['(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']
20


In [11]:
import matplotlib.pyplot as plt
ratings['rating'].value_counts().sort_index().plot(kind='bar')
plt.title('Rating Distribution')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

In [12]:
# Test TF-IDF on genres
from sklearn.feature_extraction.text import TfidfVectorizer

# Replace pipe with space
movies['genres_clean'] = movies['genres'].str.replace('|', ' ', regex=False)

tfidf = TfidfVectorizer(stop_words='english')
matrix = tfidf.fit_transform(movies['genres_clean'])

print("Matrix shape:", matrix.shape)
# Rows = number of movies, Cols = number of unique genre words

Matrix shape: (9742, 23)


In [15]:
#Test cosine similarity
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(matrix)
print("Similarity matrix shape:", similarity.shape)
# This is a movies x movies matrix
# similarity[i][j] = how similar movie i is to movie j

Similarity matrix shape: (9742, 9742)


In [ ]:
# Test a basic recommendation
def recommend_test(title, movies_df, sim_matrix, n=5):
    # Find the index of the movie
    matches = movies_df[movies_df['title'].str.lower() == title.lower()]
    if matches.empty:
        return f"Movie '{title}' not found"
    
    idx = matches.index[0]
    
    # Get similarity scores for this movie against all others
    scores = list(enumerate(sim_matrix[idx]))
    
    # Sort by score descending, skip index 0 (the movie itself)
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:n+1]
    
    # Return movie titles
    recommended = [movies_df.iloc[i[0]]['title'] for i in scores]
    return recommended

# Test it
print(recommend_test("Toy Story (1995)", movies, similarity))

['Antz (1998)', 'Toy Story 2 (1999)', 'Adventures of Rocky and Bullwinkle, The (2000)', "Emperor's New Groove, The (2000)", 'Monsters, Inc. (2001)']
